# 02 — LoRA SFT training (Stage S3, Colab T4)

Fine-tune **Llama-3.2-1B-Instruct** with LoRA on MIDI-Instruct edit shards.

Run **`01_setup_and_data.ipynb`** first (same `RUN_ID`), or mount Drive if you persisted shards there.

In [ ]:
# @title Configuration (must match notebook 01)
WORK = "/content"
MIDI_LLM = f"{WORK}/midi-llm"
RUN_ID = "pilot_v1"
RUNS_ROOT = f"{WORK}/runs/{RUN_ID}"
SHARDS_ROOT = f"{WORK}/data/shards/{RUN_ID}"

# Training knobs — T4 defaults
REPR = "remi"  # @param ["remi", "octuple", "amt"]
TRAIN_ALL_REPRS = False  # @param {type:"boolean"}
MAX_STEPS = 300  # @param {type:"integer"}
MAX_SEQ_LEN = 1024  # @param {type:"integer"}
MAX_SAMPLES = None  # @param {type:"raw"}  # None = full train shard
BATCH_SIZE = 2  # @param {type:"integer"}
GRAD_ACCUM = 4  # @param {type:"integer"}
LEARNING_RATE = 2e-4  # @param {type:"number"}
PRECISION = "fp16"  # @param ["fp16", "bf16"]  T4 → fp16

USE_DRIVE = False  # @param {type:"boolean"}
DRIVE_RUNS = "/content/drive/MyDrive/midi-llm/runs"

In [ ]:
# @title Optional — mount Google Drive for persistent checkpoints
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RUNS_ROOT = f"{DRIVE_RUNS}/{RUN_ID}"
    import os
    os.makedirs(RUNS_ROOT, exist_ok=True)
    print("Checkpoints:", RUNS_ROOT)
else:
    print("Checkpoints (ephemeral):", RUNS_ROOT)

In [ ]:
# @title Re-install if session restarted
import os
from pathlib import Path

if not Path(MIDI_LLM).is_dir():
    raise RuntimeError("Run 01_setup_and_data.ipynb first, or clone midi-llm into /content/midi-llm")

%cd {MIDI_LLM}
!pip install -q -e ".[repr,train]"

if not os.environ.get("HF_TOKEN"):
    from getpass import getpass
    from huggingface_hub import login
    os.environ["HF_TOKEN"] = getpass("HF token: ")
    login(token=os.environ["HF_TOKEN"])

In [ ]:
# @title GPU check
import torch
assert torch.cuda.is_available(), "Enable T4 GPU runtime"
print(torch.cuda.get_device_name(0))

In [ ]:
# @title Train LoRA
from pathlib import Path
from midi_llm.train.lora_sft import train_lora_sft

reprs = ["remi", "octuple", "amt"] if TRAIN_ALL_REPRS else [REPR]
results = {}
for repr_name in reprs:
    shard = Path(SHARDS_ROOT) / f"edit_{repr_name}_train.jsonl"
    if not shard.is_file():
        raise FileNotFoundError(f"Missing shard {shard} — run notebook 01")
    out_dir = Path(RUNS_ROOT) / repr_name
    print(f"\n=== Training {repr_name} → {out_dir} ===")
    train_lora_sft(
        shard,
        out_dir,
        repr_name=repr_name,
        max_steps=MAX_STEPS,
        max_seq_len=MAX_SEQ_LEN,
        max_samples=MAX_SAMPLES,
        per_device_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        precision=PRECISION,
        logging_steps=10,
        save_steps=min(200, max(50, MAX_STEPS // 2)),
    )
    results[repr_name] = str(out_dir / "lora_adapter")
    print("Saved:", results[repr_name])

print("\nAdapters:", results)

In [ ]:
# @title Zip adapters for download (if not using Drive)
import shutil
from pathlib import Path

zip_path = f"{WORK}/runs_{RUN_ID}_adapters"
shutil.make_archive(zip_path, "zip", RUNS_ROOT)
print("Download:", zip_path + ".zip")
try:
    from google.colab import files
    files.download(zip_path + ".zip")
except ImportError:
    pass

## T4 timing guide

| `MAX_STEPS` | `MAX_SEQ_LEN` | ~Time (1 arm) |
|-------------|---------------|---------------|
| 50 | 512 | 5–10 min |
| 300 | 1024 | 30–45 min |
| 3000 | 1024 | 4–6 h |

If OOM: set `BATCH_SIZE=1`, `MAX_SEQ_LEN=512`, `GRAD_ACCUM=8`.

## Next

Open **`03_eval_musicinstruct.ipynb`**.